# 🪔 Akshara-Drishti — Advanced Training Notebook\n**AI-Powered Indic Script Intelligence System**\n\nClassifies inscription images into: `brahmi`, `devanagari`, `tamil`\n\n---\n> ⚠️ **Run Cell 1 first, then restart the runtime before continuing.**

## 🛠️ Step 0: Dataset Quality Checklist\n\n**To improve model accuracy and prevent confused predictions, ensure your dataset strictly follows this criteria BEFORE training:**\n1. **Class Integrity:** Have exactly 3 folders (`brahmi`, `devanagari`, `tamil`). Delete any other folders.\n2. **Clean Data:** Remove blurry images and strictly crop images so only the script is visible.\n3. **Volume:** Ensure each class has a minimum of 500 images, and the distribution is reasonably balanced.\n4. **Diversity:** Include a mix of printed text, handwritten scripts, and stone inscriptions.

In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES  (Run once → restart runtime)
# ============================================================
!pip install -q tensorflow==2.15.0 opencv-python-headless pillow matplotlib seaborn scikit-learn

In [ ]:
# ============================================================
# CELL 2 — ALL IMPORTS
# ============================================================
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from PIL import Image

print(f"✅ TensorFlow version: {tf.__version__}")

In [ ]:
# ============================================================
# CELL 3 — MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 4 — PATHS & HYPERPARAMETERS
# ============================================================
BASE_PATH    = "/content/drive/MyDrive/AKSHARA_DHRISTI_SOFTSKILLS"
DATASET_PATH = BASE_PATH + "/dataset"
MODEL_PATH   = BASE_PATH + "/best_model.keras"   # checkpoint target
FINAL_PATH   = BASE_PATH + "/final_model.keras"  # final saved model

IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS_S1  = 20   # Stage 1 — frozen base
EPOCHS_S2  = 10   # Stage 2 — fine-tune
SEED       = 123

In [ ]:
# ============================================================
# CELL 5 — LOAD DATASET (Strictly enforcing 3 classes)
# ============================================================
# We explicitly set class_names to filter out any unwanted 'script_1' folders
ALLOWED_CLASSES = ["brahmi", "devanagari", "tamil"]

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_names=ALLOWED_CLASSES,   # <-- ENFORCEMENT
    label_mode='categorical'       # <-- Required for label smoothing later
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_names=ALLOWED_CLASSES,   # <-- ENFORCEMENT
    label_mode='categorical'
)

class_names = train_ds.class_names
print(f"\n✅ STRICTLY LOADED CLASSES: {class_names}")
assert class_names == ALLOWED_CLASSES, "Error: Classes loaded do not match target!"

In [ ]:
# ============================================================
# CELL 6 — PREPROCESSING (Matches Backend Option A Exactly)
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

# We REMOVED CLAHE & Edge detection here so the training distribution
# perfectly matches the standard prediction path in the backend app.
# Simply cast and normalize to [0, 1].
train_ds = train_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTOTUNE)

# Cache → Shuffle → Prefetch
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

print("✅ Native preprocessing pipeline ready")

In [ ]:
# ============================================================
# CELL 7 — DATA AUGMENTATION
# ============================================================
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
    layers.RandomFlip("horizontal"),
], name="augmentation")

In [ ]:
# ============================================================
# CELL 8 — BUILD IMPROVED MODEL
# ============================================================
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # frozen for Stage 1

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = data_augmentation(inputs)
x       = base_model(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)

# ----- IMPROVED CLASSIFICATION HEAD -----
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
# ----------------------------------------

outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = Model(inputs, outputs, name="AksharaDrishti-V2")
model.summary()

In [ ]:
# ============================================================
# CELL 9 — CALLBACKS & COMPILE
# ============================================================
checkpoint = ModelCheckpoint(MODEL_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)

early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

# Reduce learning rate if training halts, giving it a better chance to converge nicely
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

callbacks = [checkpoint, early_stop, reduce_lr]

# Using Label Smoothing to prevent confident misclassifications
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
print("✅ Compilation ready with Label Smoothing")

In [ ]:
# ============================================================
# CELL 10 — STAGE 1: TRAIN (Frozen Base)
# ============================================================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_S1,
    callbacks=callbacks
)

In [ ]:
# ============================================================
# CELL 11 — STAGE 2: FINE-TUNE
# ============================================================
base_model.trainable = True

# Fine-tune the last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_S2,
    callbacks=callbacks
)

In [ ]:
# ============================================================
# CELL 12 — PLOT RESULTS
# ============================================================
final_train_acc = history_fine.history['accuracy'][-1] * 100
final_val_acc = history_fine.history['val_accuracy'][-1] * 100
print(f"\n🏆 Final Training Accuracy:   {final_train_acc:.2f}%")
print(f"🏆 Final Validation Accuracy: {final_val_acc:.2f}%\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_fine.history['accuracy'],     label='Train', color='royalblue')
axes[0].plot(history_fine.history['val_accuracy'], label='Validation', color='gold')
axes[0].set_title('Fine-Tune Accuracy')
axes[0].legend()

axes[1].plot(history_fine.history['loss'],     label='Train', color='royalblue')
axes[1].plot(history_fine.history['val_loss'], label='Validation', color='gold')
axes[1].set_title('Fine-Tune Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 13 — EVALUATION: CONFUSION MATRIX & REPORT
# ============================================================
print("Generating evaluation metrics...")
y_true = []
y_pred = []

# Run prediction on validation set
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

# Classification Report
report = classification_report(y_true, y_pred, target_names=class_names)
print("\nClassification Report:")
print(report)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ============================================================
# CELL 14 — SAVE FINAL MODEL
# ============================================================
model.save(FINAL_PATH)
print(f"✅ Final model saved to: {FINAL_PATH}")
print(f"\n👉 Take {FINAL_PATH} and put it inside your website backend `model/` directory!")